<a href="https://colab.research.google.com/github/kanzuleman-ai/AI-Chatbot-Day1/blob/main/AI-Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 AI Chatbot — Day 1

A basic AI chatbot built with Python and Google's Gemini API.

### Features
- AI-powered responses
- Conversation history
- Context-aware replies
- Continuous conversation
- Simple and beginner-friendly interface
- Exit command

### Technologies
- Python
- Google Gemini API
- Google Colab

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini connected successfully!")

In [ ]:
print("🤖 AI Chatbot")
print("Hello! I'm your AI chatbot, powered by Gemini.")
print("I'm here to answer your questions and help you out.")
print("Type 'exit' to end the conversation.\n")

chat_history = []

while True:
    user_message = input("You: ")

    if user_message.lower() == "exit":
        print("Bot: Goodbye! 👋")
        break

    chat_history.append(f"User: {user_message}")

    conversation = "\n".join(chat_history)

    response = client.models.generate_content(
        model="models/gemini-3.6-flash",
        contents=conversation
    )

    bot_reply = response.text

    print("Bot:", bot_reply)
    print()

    chat_history.append(f"Bot: {bot_reply}")

# ⚙️ Day 2 — AI Chatbot Development

In Day 2, the basic chatbot was improved to make it more structured, reliable, and user-friendly.

### Day 2 Improvements

- Added a defined system prompt for the AI's role and personality
- Added loading state while generating responses
- Added error handling for API failures
- Added input validation
- Improved the chatbot interface
- Maintained secure API key handling using Google Colab Secrets

The goal was to move from a basic API-based chatbot to a more complete and reliable chatbot application.

In [ ]:
# ============================================
# DAY 2 + DAY 3 — FINAL AI CHATBOT
# ============================================

from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

# ============================================
# 1. SYSTEM PROMPT
# ============================================

SYSTEM_PROMPT = """
You are Nova, a helpful, friendly, and professional AI assistant.

Your responsibilities are:
- Give clear, accurate, and useful answers.
- Explain difficult topics in simple language.
- Maintain a polite and conversational personality.
- Use previous conversation context when relevant.
- Do not invent information when you are unsure.
- Keep answers reasonably concise unless the user asks for detail.
- Format answers clearly using Markdown when appropriate.

Always prioritize accuracy, clarity, and helpfulness.
"""


# ============================================
# 2. CONVERSATION HISTORY
# ============================================

conversation_history = []


# ============================================
# 3. PROMPT STRUCTURE + CONTEXT
# ============================================

def build_prompt(user_message):

    history_text = ""

    if conversation_history:

        history_text = "\nPrevious conversation:\n"

        for message in conversation_history:

            role = message["role"].capitalize()
            content = message["content"]

            history_text += f"{role}: {content}\n"

    prompt = f"""
{SYSTEM_PROMPT}

{history_text}

Current user message:
{user_message}

Use the previous conversation when it is relevant.
Respond naturally, clearly, and helpfully.
"""

    return prompt


# ============================================
# 4. AI RESPONSE + ERROR HANDLING
# ============================================

def get_ai_response(user_message):

    try:

        prompt = build_prompt(user_message)

        response = client.models.generate_content(
            model="models/gemini-3.6-flash",
            contents=prompt
        )

        if not response or not response.text:
            raise ValueError("The AI returned an empty response.")

        return response.text.strip()

    except Exception as e:

        return f"ERROR: {str(e)}"


# ============================================
# 5. CHAT OUTPUT AREA
# ============================================

chat_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #cccccc",
        padding="15px",
        height="400px",
        overflow="auto"
    )
)


# ============================================
# 6. USER INPUT
# ============================================

user_input = widgets.Textarea(
    placeholder="Type your message here...",
    layout=widgets.Layout(
        width="80%",
        height="70px"
    )
)


# ============================================
# 7. BUTTONS
# ============================================

send_button = widgets.Button(
    description="Send",
    button_style="primary",
    icon="paper-plane"
)

clear_button = widgets.Button(
    description="Clear Chat",
    button_style="warning",
    icon="trash"
)


# ============================================
# 8. DISPLAY MESSAGES
# ============================================

def display_message(role, message):

    with chat_output:

        if role == "user":

            display(
                Markdown(
                    f"**You:** {message}"
                )
            )

        else:

            display(
                Markdown(
                    f"**Nova:**\n\n{message}"
                )
            )


# ============================================
# 9. SEND MESSAGE
# ============================================

def send_message(button):

    message = user_input.value.strip()

    # Input validation
    if not message:

        with chat_output:

            display(
                Markdown(
                    "⚠️ **Please enter a message first.**"
                )
            )

        return

    # Display user message
    display_message("user", message)

    # Clear input field
    user_input.value = ""

    # Loading state
    with chat_output:

        loading = display(
            Markdown("⏳ **Nova is thinking...**"),
            display_id=True
        )

    # Get AI response
    bot_reply = get_ai_response(message)

    # Remove loading state
    loading.update("")

    # Handle API errors
    if bot_reply.startswith("ERROR:"):

        with chat_output:

            display(
                Markdown(
                    f"⚠️ **Something went wrong:**\n\n"
                    f"{bot_reply}"
                )
            )

        return

    # Save user message
    conversation_history.append(
        {
            "role": "user",
            "content": message
        }
    )

    # Save AI response
    conversation_history.append(
        {
            "role": "assistant",
            "content": bot_reply
        }
    )

    # Display AI response as Markdown
    display_message("assistant", bot_reply)


# ============================================
# 10. CLEAR CHAT
# ============================================

def clear_chat(button):

    conversation_history.clear()

    with chat_output:

        clear_output()

        display(
            Markdown(
                "🧹 **Chat cleared successfully.**\n\n"
                "You can start a new conversation."
            )
        )


# ============================================
# 11. BUTTON EVENTS
# ============================================

send_button.on_click(send_message)

clear_button.on_click(clear_chat)


# ============================================
# 12. FINAL USER INTERFACE
# ============================================

display(
    Markdown(
        """
# 🤖 Nova AI Chatbot

Welcome! I'm **Nova**, your AI assistant.

Ask me anything and I'll try my best to help you.
"""
    )
)

display(chat_output)

display(
    widgets.HBox(
        [
            user_input,
            widgets.VBox(
                [
                    send_button,
                    clear_button
                ]
            )
        ]
    )
)

# 🚀 Day 3 — Further Improvements

In Day 3, the chatbot was further improved to make the conversation more interactive, structured, and user-friendly.

### Day 3 Improvements

* **Conversation History & Context:** The chatbot keeps track of previous messages and uses relevant conversation context when generating responses.
* **Clear Chat:** Added a Clear Chat button to remove the current conversation history and start a fresh conversation.
* **Better Prompt Structure:** User messages, previous conversation, and system instructions are organized into a structured prompt.
* **Markdown Responses:** AI responses are displayed with Markdown formatting for better readability.
* **Input Validation:** The chatbot checks for empty messages before sending a request to the API.
* **Improved Response & Error Handling:** API errors and empty responses are handled without crashing the chatbot.
* **Improved User Interface:** Added a clean interface with a message area, input box, Send button, and Clear Chat button.

### Final Result

The final chatbot combines the Day 2 reliability improvements with the Day 3 functionality improvements into one complete working application.